# 08 — PPO with Symbolic (RAM) Observations

PPO agent using the RAM-based symbolic grid (13x16) instead of pixels.

**Setup:**
- Observation: flattened 13x16 grid with 4 frame stack = 832-dim vector
- Policy: MlpPolicy with [512, 512] hidden layers
- Much faster than pixel-based (no CNN, no image processing)
- Same PPO hyperparameters as pixel version

In [1]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

/home/contente/Documents/ENSTA/autonomous/CSC-52081-ContinousMountainCar


In [2]:
import torch
from stable_baselines3 import PPO

from src.wrappers import make_symbolic_vec_env, make_symbolic_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import PPOConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using CUDA: True
GPU: NVIDIA GeForce GTX 1650


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
# Create 8 parallel symbolic environments
NUM_ENVS = 8

env = make_symbolic_vec_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4,
    n_stack=4,
    flatten=True,
    num_envs=NUM_ENVS,
)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')
print(f'Parallel envs: {NUM_ENVS}')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Observation space: (832,)
Action space: 7
Parallel envs: 8


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


In [4]:
# Phase 1: Train from scratch
config = PPOConfig(
    policy='MlpPolicy',
    learning_rate=2.5e-4,
    n_steps=512,
    batch_size=256,
    n_epochs=4,
    gamma=0.9,           # vietnh1009 uses 0.9
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    total_timesteps=2_000_000,
)

model = PPO(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    n_steps=config.n_steps,
    batch_size=config.batch_size,
    n_epochs=config.n_epochs,
    gamma=config.gamma,
    gae_lambda=config.gae_lambda,
    clip_range=config.clip_range,
    ent_coef=config.ent_coef,
    vf_coef=config.vf_coef,
    max_grad_norm=config.max_grad_norm,
    policy_kwargs=dict(net_arch=[512, 512]),
    tensorboard_log='logs/symbolic_ppo',
    verbose=1,
    device='cpu',  # MLP is faster on CPU
)

print(f'Phase 1: {config.total_timesteps:,} timesteps')
print(f'Device: {model.device}')
print(f'Batch per rollout: {config.n_steps * NUM_ENVS} samples')

Using cpu device
Phase 1: 2,000,000 timesteps
Device: cpu
Batch per rollout: 4096 samples


In [5]:
%load_ext tensorboard
%tensorboard --logdir logs/symbolic_ppo

In [6]:
# Phase 1: Train for 2M steps
callback = CheckpointAndLogCallback(
    save_path='models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=1,
)
model.save('models/symbolic_ppo/phase1_model')
print('Phase 1 complete!')

Logging to logs/symbolic_ppo/PPO_8


/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional infor

-----------------------------
| time/              |      |
|    fps             | 526  |
|    iterations      | 1    |
|    time_elapsed    | 7    |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 268         |
|    ep_rew_mean          | 56.4        |
|    flag_rate_100        | 0           |
| time/                   |             |
|    fps                  | 494         |
|    iterations           | 2           |
|    time_elapsed         | 16          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.013492234 |
|    clip_fraction        | 0.15        |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.93       |
|    explained_variance   | -0.00964    |
|    learning_rate        | 0.00025     |
|    loss                 | 0.367       |
|    n_updates            | 4     

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 183         |
|    ep_rew_mean          | 116         |
|    flag_rate_100        | 0.01        |
| time/                   |             |
|    fps                  | 495         |
|    iterations           | 30          |
|    time_elapsed         | 247         |
|    total_timesteps      | 122880      |
| train/                  |             |
|    approx_kl            | 0.014601193 |
|    clip_fraction        | 0.163       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.23       |
|    explained_variance   | 0.656       |
|    learning_rate        | 0.00025     |
|    loss                 | 1.11        |
|    n_updates            | 116         |
|    policy_gradient_loss | -0.0162     |
|    value_loss           | 2.71        |
-----------------------------------------
-----------------------------------------
| time/                   |       

...

-----------------------------------------
| time/                   |             |
|    fps                  | 500         |
|    iterations           | 48          |
|    time_elapsed         | 393         |
|    total_timesteps      | 196608      |
| train/                  |             |
|    approx_kl            | 0.020570995 |
|    clip_fraction        | 0.199       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.06       |
|    explained_variance   | 0.81        |
|    learning_rate        | 0.00025     |
|    loss                 | 0.575       |
|    n_updates            | 188         |
|    policy_gradient_loss | -0.0131     |
|    value_loss           | 1.55        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 241         |
|    ep_rew_mean          | 164         |
|    flag_rate_100        | 0.07        |
| time/                   |       

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 261         |
|    ep_rew_mean          | 180         |
|    flag_rate_100        | 0.14        |
| time/                   |             |
|    fps                  | 500         |
|    iterations           | 51          |
|    time_elapsed         | 417         |
|    total_timesteps      | 208896      |
| train/                  |             |
|    approx_kl            | 0.016990144 |
|    clip_fraction        | 0.183       |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.03       |
|    explained_variance   | 0.839       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.472       |
|    n_updates            | 200         |
|    policy_gradient_loss | -0.0146     |
|    value_loss           | 1.35        |
-----------------------------------------
-----------------------------------------
| time/                   |       

/home/contente/miniconda3/envs/mario_lucas/lib/python3.14/site-packages/gym_super_mario_bros/smb_env.py:177: RuntimeWarning: overflow encountered in scalar add
  return 255 + (255 - self._y_pixel)


-----------------------------------------
| time/                   |             |
|    fps                  | 500         |
|    iterations           | 54          |
|    time_elapsed         | 442         |
|    total_timesteps      | 221184      |
| train/                  |             |
|    approx_kl            | 0.026098453 |
|    clip_fraction        | 0.244       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.969      |
|    explained_variance   | 0.811       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.731       |
|    n_updates            | 212         |
|    policy_gradient_loss | -0.0105     |
|    value_loss           | 1.53        |
-----------------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 263         |
|    ep_rew_mean          | 180         |
|    flag_rate_100        | 0.17        |
| time/                   |       

In [ ]:
# Phase 2: Load best model, lower lr, train 2M more
from stable_baselines3 import PPO

model = PPO.load('models/symbolic_ppo/phase1_model', env=env, device='cpu')
model.learning_rate = 1e-5

callback2 = CheckpointAndLogCallback(
    save_path='models/symbolic_ppo',
    save_freq=50_000,
)

model.learn(
    total_timesteps=2_000_000,
    callback=callback2,
    log_interval=1,
)
model.save('models/symbolic_ppo/final_model')
print('Phase 2 complete!')

Logging to logs/symbolic_ppo/PPO_9
-----------------------------
| time/              |      |
|    fps             | 580  |
|    iterations      | 1    |
|    time_elapsed    | 7    |
|    total_timesteps | 4096 |
-----------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 325         |
|    ep_rew_mean          | 299         |
|    flag_rate_100        | 0.857       |
| time/                   |             |
|    fps                  | 551         |
|    iterations           | 2           |
|    time_elapsed         | 14          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.020349072 |
|    clip_fraction        | 0.135       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.858      |
|    explained_variance   | 0.887       |
|    learning_rate        | 0.00025     |
|    loss                 | 0.0459      |

In [ ]:
# Plot training results
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

rewards = callback.episode_rewards + (callback2.episode_rewards if 'callback2' in dir() else [])
lengths = callback.episode_lengths + (callback2.episode_lengths if 'callback2' in dir() else [])
flags = callback.episode_flags + (callback2.episode_flags if 'callback2' in dir() else [])

if len(rewards) > 0:
    window = min(100, len(rewards))

    ax = axes[0]
    ax.plot(rewards, alpha=0.3, color='blue')
    if len(rewards) > window:
        smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(rewards)), smoothed, color='blue', linewidth=2)
    ax.set_title('Episode Rewards')
    ax.set_xlabel('Episode')

    ax = axes[1]
    ax.plot(lengths, alpha=0.3, color='orange')
    if len(lengths) > window:
        smoothed = np.convolve(lengths, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(lengths)), smoothed, color='orange', linewidth=2)
    ax.set_title('Episode Lengths')
    ax.set_xlabel('Episode')

    ax = axes[2]
    if len(flags) > 0:
        cumulative_flags = np.cumsum(flags) / (np.arange(len(flags)) + 1)
        ax.plot(cumulative_flags, color='green', linewidth=2)
    ax.set_title('Cumulative Flag Rate')
    ax.set_xlabel('Episode')
    ax.set_ylim(0, 1)

    plt.tight_layout()
    plt.savefig('models/symbolic_ppo/training_curves.png', dpi=150)
    plt.show()
else:
    print('No episode data collected yet.')

In [ ]:
# Evaluate final model
from stable_baselines3 import PPO
import numpy as np

eval_model = PPO.load('models/symbolic_ppo/final_model')

eval_env = make_symbolic_env(
    env_id='SuperMarioBros-1-1-v3',
    skip=4, n_stack=4, flatten=True,
)

rewards = []
lengths = []
flags = []

for ep in range(10):
    reset_result = eval_env.reset()
    obs = reset_result[0] if isinstance(reset_result, tuple) else reset_result
    done = False
    total_reward = 0.0
    steps = 0
    flag = False

    while not done:
        action, _ = eval_model.predict(obs, deterministic=True)
        step_result = eval_env.step(int(action))
        if len(step_result) == 5:
            obs, reward, terminated, truncated, info = step_result
            done = terminated or truncated
        else:
            obs, reward, done, info = step_result
        total_reward += float(reward)
        steps += 1
        if isinstance(info, dict) and info.get('flag_get', False):
            flag = True

    rewards.append(total_reward)
    lengths.append(steps)
    flags.append(flag)
    status = 'FLAG!' if flag else 'DEAD'
    print(f'Episode {ep+1}: reward={total_reward:.1f}, steps={steps}, {status}')

print(f'\nMean reward: {np.mean(rewards):.1f} +/- {np.std(rewards):.1f}')
print(f'Mean length: {np.mean(lengths):.0f}')
print(f'Flag rate: {np.mean(flags):.0%}')

eval_env.close()